### In this notebook we show how the model to model evaluation module works

In [17]:
import sys
import os
import json

sys.path.append("../")
sys.path.append("../model_evaluation/")

from model_evaluation.BPMN_conversion import BPMNConverter, XMLBPMNConverter

In [18]:
def load_model(path):
    """Load a BPMN model from a .json (Signavio) or .bpmn/.xml (BPMN 2.0) file.
    Returns the normalised dict ready for the evaluation pipeline.
    """
    if path.endswith(".xml") or path.endswith(".bpmn"):
        return XMLBPMNConverter.convert_file(path).to_dict()
    else:
        with open(path, "r", encoding="utf-8") as fh:
            raw = json.load(fh)
        return BPMNConverter.convert(raw).to_dict()


# Load the first model (JSON or BPMN)
path_model1 = "../examples/01_BPMN_Training-T-shirt_order_simple.bpmn"

# Load the second model (JSON or BPMN)
path_model2 = "../examples/02_BPMN_Training-T-shirt_order_extended.bpmn"


model_1_json = load_model(path_model1)
model_2_json = load_model(path_model2)

In [19]:
# ==============================================================================
# BPMN Model Comparison Pipeline
# ==============================================================================
import json

# from rendering import create_similarity_dashboard, print_similarity_report
from bpmn_normalization import normalize_atomic_names
from bpmn_similarity import calculate_bpmn_similarity
from utils import cosine_sim_optimized

print("BPMN MODEL COMPARISON PIPELINE")


# Step 1: Model Summary
print("\n[1] MODEL STATISTICS")


def count_elements(model):
    """Count BPMN elements in a model."""
    return {
        "activities": len(model.get("activities", [])),
        "events": len(model.get("events", [])),
        "gateways": len(model.get("gateways", [])),
        "sequence_flows": len(model.get("sequenceFlows", [])),
        "message_flows": len(model.get("messageFlows", [])),
        "pools": len(model.get("pools", [])),
        "lanes": sum(len(p.get("lanes", [])) for p in model.get("pools", [])),
    }


model1_counts = count_elements(model_1_json)
model2_counts = count_elements(model_2_json)

print(f"Model 1: {sum(model1_counts.values())} total elements")
for key, val in model1_counts.items():
    if val > 0:
        print(f"  • {key.replace('_', ' ').title()}: {val}")

print(f"\nModel 2: {sum(model2_counts.values())} total elements")
for key, val in model2_counts.items():
    if val > 0:
        print(f"  • {key.replace('_', ' ').title()}: {val}")

# Step 2: Normalize Names
print("\n[2] SEMANTIC NAME NORMALIZATION")

threshold = 0.6
print(f"Aligning element names using a sentence transformer model (threshold={threshold})...")

model2_aligned, mappings = normalize_atomic_names(model_1_json, model_2_json, cosine_sim_optimized, threshold=threshold)

from bpmn_sets import extract_bpmn_sets

# print("model1:\n", extract_bpmn_sets(model_1_json), "\nmodel2_aligned:\n", extract_bpmn_sets(model2_aligned))


total_mappings = sum(len(v) for v in mappings.values())


if total_mappings > 0:
    print(f"✓ Applied {total_mappings} semantic name mappings")
    for elem_type, mapping in mappings.items():
        if mapping:
            print(f"  • {elem_type}: {len(mapping)} mappings")
            # Show first example
            first_old, first_new = next(iter(mapping.items()))
            print(f"    Example: '{first_old}' → '{first_new}'")
else:
    print("✓ No mappings needed (names already aligned)")

tr1 = extract_traces(model_1_json, timeout_seconds=5, max_loop_depth=3)

tr2 = extract_traces(model2_aligned, timeout_seconds=5, max_loop_depth=3)

print(f"\nExample traces from model 1 ...\n")
print(tr1.all_traces()[:3])

print(f"\nExample traces from model 2 ...\n")
print(tr2.all_traces()[:3])

# Step 3: Calculate Similarity Without Normalization
print("\n[3] SIMILARITY ANALYSIS")


# similarity_without_norm = calculate_bpmn_similarity(model_1_json, model_2_json, method="dice")
from bpmn_similarity import calculate_bpmn_similarity, calculate_trace_similarity, calculate_hybrid_similarity
from trace_extraction import extract_traces

struct = calculate_bpmn_similarity(model_1_json, model2_aligned, method="jaccard")

beh = calculate_trace_similarity(tr1, tr2, method="jaccard")
hybrid = calculate_hybrid_similarity(struct, beh, structural_weight=0.5)
print(hybrid)

BPMN MODEL COMPARISON PIPELINE

[1] MODEL STATISTICS
Model 1: 26 total elements
  • Activities: 7
  • Events: 2
  • Gateways: 2
  • Sequence Flows: 11
  • Pools: 1
  • Lanes: 3

Model 2: 31 total elements
  • Activities: 9
  • Events: 2
  • Gateways: 2
  • Sequence Flows: 13
  • Pools: 1
  • Lanes: 4

[2] SEMANTIC NAME NORMALIZATION
Aligning element names using a sentence transformer model (threshold=0.6)...
✓ Applied 14 semantic name mappings
  • activity_names: 7 mappings
    Example: 'Ship Goods' → 'Ship Goods'
  • event_names: 2 mappings
    Example: 'T-shirt order  received' → 'T-shirt order  received'
  • gateway_names: 1 mappings
    Example: 'Custom print order?' → 'Custom print order?'
  • pool_names: 1 mappings
    Example: 'FairTrade T-Shirt Company' → 'FairTrade T-Shirt Company'
  • lane_names: 3 mappings
    Example: 'Printing' → 'Printing'

Example traces from model 1 ...

[['T-shirt order  received', 'Receive Customer Order', 'Receive Payment', 'Send T-shirt to Printing 

In [21]:
from rendering.dashboard import BPMNSimilarityDashboard
from bpmn_normalization import normalize_atomic_names
from bpmn_similarity import (
    calculate_bpmn_similarity,
    calculate_trace_similarity,
    calculate_hybrid_similarity,
)
from trace_extraction import extract_traces
from utils.string_similarity import cosine_sim_optimized

dashboard = BPMNSimilarityDashboard(
    model_1_json,  # raw
    model_2_json,  # raw, NOT pre-aligned
    similarity_func=cosine_sim_optimized,
    calculate_similarity_func=calculate_bpmn_similarity,
    normalize_func=normalize_atomic_names,
    extract_traces_func=extract_traces,
    calculate_trace_similarity_func=calculate_trace_similarity,
    calculate_hybrid_func=calculate_hybrid_similarity,
    initial_threshold=0.7,
)
dashboard.display()